# Medicare Archive Data



## All Archives


- https://data.cms.gov/provider-data/archived-data/doctors-clinicians


In [ ]:
%%bash

for i in {2019..2026} ; do
  echo == ${i}
  curl -O -L https://data.cms.gov/provider-data/sites/default/files/dataset-archives/theme/doctors-clinicians/doctors-clinicians_annual_${i}.zip
done


In [ ]:
%%bash

for i in {2019..2026} ; do
  (
    echo == ${i}
    mkdir -p zips/${i}
    cd zips/${i}
    pwd
    unzip ../doctors-clinicians_annual_${i}.zip
  )
done


# Gone


From [Doctors and clinicians datasets](https://data.cms.gov/provider-data/search?theme=Doctors%20and%20clinicians) page.

- [National Downloadable File]( https://data.cms.gov/provider-data/dataset/mj5m-pzi6 )
- [Data Dictionary]( https://data.cms.gov/provider-data/sites/default/files/data_dictionaries/physician/DOC_Data_Dictionary.pdf )


From the National Downloadable File URL, we see that the dataset ID is `mj5m-pzi6`.  We can use that to query the CMS API to get the link to the CSV file.



In [ ]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
import requests
import pandas as pd
import duckdb
from google.colab import userdata
import os


## Query the CMS API


In [ ]:
dataset_id = "mj5m-pzi6"
url = f"https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/{dataset_id}"

dataset_id, url

In [ ]:
response = requests.get(url)
response


In [ ]:
metadata = response.json()
metadata


In [ ]:
url = metadata["distribution"][0]["downloadURL"]
url


## Download the CSV file


In [ ]:
! curl -I -L -s {url}


In [ ]:
! curl -L -O {url}


In [ ]:
csv_file = url.split("/")[-1]
csv_file


In [ ]:
! ls -la {csv_file}


In [ ]:
! du -ms {csv_file}


In [ ]:
! wc -l {csv_file}

## DuckDB: query a CSV file



In [ ]:
duckdb.query(f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
  LIMIT 10
""").show() ;


## DuckDB: query a CSV file with a CTE


In [ ]:
duckdb.query(f"""
  WITH
  medicare as (
    SELECT *
    FROM read_csv(
      "{csv_file}",
      types={{'Telephone Number': 'VARCHAR'}},
      auto_detect=True
    )
  ),

  medicare_nm as (
    SELECT *
    FROM medicare
    WHERE State = 'NM'
  ),

  medicare_abq as (
    SELECT *
    FROM medicare_nm
    WHERE "City/Town" IN ['ALBUQERQUE', 'RIO RANCHO']
  )

  SELECT *
  FROM medicare_abq
  ORDER BY "City/Town", NPI
  LIMIT 10
""").show()


In [ ]:
query = f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
""".strip()
print(query)


In [ ]:
rel = duckdb.sql(query)


## Convert CSV to single parquet file


In [ ]:
parquet_file = csv_file.replace(".csv",".parquet")
parquet_file


In [ ]:
export_query = f'''
  COPY rel TO '{parquet_file}'
  (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
'''
print(export_query)


In [ ]:
duckdb.sql(export_query)


In [ ]:
! du -ms {parquet_file} {csv_file}


## Verify that DuckDB can query the local parquet file


In [ ]:
query = f"SELECT * FROM read_parquet('{parquet_file}')"
query


In [ ]:
rel = duckdb.sql(query)


In [ ]:
print(f"Total rows found: {len(rel):,}")


In [ ]:
rel.show()


## Convert CSV to hive-partitioned parquet


In [ ]:
parquet_path = "medicare_DAC_national"
parquet_path


In [ ]:
export_query = f'''
  COPY rel TO '{parquet_path}'
  (FORMAT PARQUET, PARTITION_BY (State), OVERWRITE_OR_IGNORE 1);
'''
print(export_query)


In [ ]:
duckdb.sql(export_query)


In [ ]:
! tree {parquet_path}


In [ ]:
! du -ms {parquet_path} {parquet_file} {csv_file}


## Verify that DuckDB can query hive parquet files


In [ ]:
dataset_path = f"{parquet_path}/**/*.parquet"
dataset_path


In [ ]:
query = f"SELECT * FROM read_parquet('{dataset_path}')"
query


In [ ]:
rel = duckdb.sql(query)


In [ ]:
print(f"Total rows found across all partitions: {len(rel):,}")


In [ ]:
rel.show()


## Push to Hugging Face


In [ ]:
stop


In [ ]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]} "

In [ ]:
# Log in automatically without an interactive prompt
!hf auth login --token $HF_TOKEN


### Push CSV file


In [ ]:
os.environ["hf_csv"] = csv_file
! echo ${hf_csv}


In [ ]:
%%capture hf_upload_csv
%%bash

# echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ${hf_csv} \
  ${hf_csv}


In [ ]:
print(hf_upload_csv.stdout)


### Push Parquet File

In [ ]:
os.environ["hf_parquet"] = parquet_file
! echo ${hf_parquet}


In [ ]:
%%capture hf_upload_parquet
%%bash

# echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ./${hf_parquet} \
  ${hf_parquet}


In [ ]:
print(hf_upload_parquet.stdout)


### Push Hive Parquet Files

In [ ]:
os.environ["hf_hive"] = parquet_path
! echo ${hf_hive}


In [ ]:
%%capture hf_upload_hive
%%bash

# echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ./${hf_hive} \
  ${hf_hive}


In [ ]:
print(hf_upload_hive.stdout)


## Query Hugging Face using DuckDB


### CSV File


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_csv = f"hf://datasets/rwcitek/medicare/{csv_file}"
hf_url_csv


In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(1)
  FROM read_csv('{hf_url_csv}')
"""
print(query)


In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


### Parquet File


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_parquet = f"hf://datasets/rwcitek/medicare/{parquet_file}"
hf_url_parquet


In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(1)
  FROM read_parquet('{hf_url_parquet}')
"""
print(query)


In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


In [ ]:
# Query the remote dataset directly
query = f"""
  DESCRIBE
  SELECT *
  FROM read_parquet('{hf_url_parquet}')
"""
print(query)


In [ ]:
duckdb.sql(query).show(max_rows=100)


### Hive Parquet Files


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_hive = f"hf://datasets/rwcitek/medicare/{parquet_path}/**/*.parquet"
hf_url_hive


In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(*)
  FROM read_parquet('{hf_url_hive}')
"""


In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


In [ ]:
# Query the remote dataset directly
query = f"""
  DESCRIBE
  SELECT *
  FROM read_parquet('{hf_url_hive}')
"""
print(query)


In [ ]:
duckdb.sql(query).show(max_rows=100)
